> **Note for the repo copy.** Outputs are cleared to keep this file small. The raw data (about 190 MB) is not in this repository. It comes from the [Eviction Lab](https://evictionlab.org/), the City of Philadelphia L+I violations file, [OpenDataPhilly](https://www.opendataphilly.org/) and Landsat 8. The interactive version of Part 1 is in `index.html` next to this notebook.

# Assignment 3

This assignment will contain two parts:

1. Exploring evictions and code violations in Philadelphia
2. Comparing the NDVI in Philadelphia


Download all the necessary datasets from [here](assignment3-data.zip) before starting the assignment. Please put the datasets in a folder named `data` in your working directory with your `.ipynb` file.

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import warnings


# See lots of columns
pd.options.display.max_columns = 999

# Hide warnings due to issue in shapely package 
# See: https://github.com/shapely/shapely/issues/1345
np.seterr(invalid="ignore");
warnings.filterwarnings("ignore")

## Part 1: Exploring Evictions and Code Violations in Philadelphia

In this assignment, we'll explore spatial trends evictions in Philadelphia using data from the [Eviction Lab](https://evictionlab.org/) and building code violations using data from [OpenDataPhilly](https://www.opendataphilly.org/).

We'll be exploring the idea that evictions can occur as retaliation against renters for reporting code violations. Spatial correlations between evictions and code violations from the City's Licenses and Inspections department can offer some insight into this question. 

**A couple of interesting background readings:**
- [HuffPost article](https://www.huffingtonpost.com/entry/cities-are-starting-to-pay-attention-to-the-eviction-crisis-thats-devastated-poor-tenants_us_5b1a7b21e4b0bbb7a0dbd59e)
- [PlanPhilly article](http://planphilly.com/articles/2018/04/12/philly-landlords-evict-more-people-than-owners-in-other-large-cities)

## 1.1 Explore Eviction Lab Data

The Eviction Lab built the first national database for evictions. If you aren't familiar with the project, you can explore their website: https://evictionlab.org/ 

### 1.1.1 Read data using `geopandas`

The first step is to read the eviction data by census tract using `geopandas`. The data for all of Pennsylvania by census tract is available in the `data/` folder in a GeoJSON format.

Load the data file "PA-tracts.geojson" using `geopandas`

**Note:** If you'd like to see all columns in the data frame, you can increase the max number of columns using pandas display options:

In [ ]:
PA_tracts = gpd.read_file("../data/assignment3-data/PA-tracts.geojson")

In [ ]:
PA_tracts.head(5)

### 1.1.2 Explore and trim the data 

We will need to trim data to Philadelphia only. Take a look at the data dictionary for the descriptions of the various columns in top-level repository folder: `eviction_lab_data_dictionary.txt`

**Note:** the column names are shortened — see the end of the above file for the abbreviations. The numbers at the end of the columns indicate the years. For example, `e-16` is the number of evictions in 2016. 

Take a look at the individual columns and trim to census tracts in Philadelphia. (**Hint:** Philadelphia is both a city and a county).

In [ ]:
philly_county = PA_tracts[PA_tracts["pl"] == 'Philadelphia County, Pennsylvania']
philly_county.head()

### 1.1.3 Transform from wide to tidy format

For this assignment, we are interested in the number of evictions by census tract for various years. Right now, each year has it's own column, so it will be easiest to transform to a tidy format. 

Use the `pd.melt()` function to transform the eviction data into tidy format, using the number of evictions from **2003 to 2016**.

The tidy data frame should have four columns: `GEOID`, `geometry`, a column holding the number of evictions, and a column telling you what the name of the original column was for that value.

**Hints:**
- You'll want to specify the `GEOID` and `geometry` columns as the `id_vars`. This will keep track of the census tract information. 
- You should specify the names of the columns holding the number of evictions as the `value_vars`.
- You can generate a list of this column names using Python's f-string formatting:
    ```python
    value_vars = [f"e-{x:02d}" for x in range(3, 17)]
    ```

In [ ]:
philly_county.columns

In [ ]:
value_vars = [f"e-{x:02d}" for x in range(3, 17)]
print(value_vars)

In [ ]:
evictions_tidy = pd.melt(
    philly_county,
    id_vars=["GEOID", "geometry"],
    value_vars = value_vars,
        var_name= "Date",
        value_name= "evictions",
    )
evictions_tidy.head()

### 1.1.4 Plot the total number of evictions per year from 2003 to 2016

Use `hvplot` to plot the total number of evictions from 2003 to 2016. You will first need to perform a group by operation and sum up the total number of evictions for all census tracts, and then use `hvplot()` to make your plot.

You can use any type of `hvplot` chart you'd like to show the trend in number of evictions over time.

In [ ]:
import holoviews as hv
import hvplot.pandas

hv.extension("bokeh")

In [ ]:
evictions_by_year = evictions_tidy.groupby(["Date"], as_index=False)["evictions"].sum()
evictions_by_year.head()

In [ ]:
evictions_by_year["Year"] = 2000 + evictions_by_year["Date"].str[2:].astype(int)

In [ ]:
evictions_by_year_plot = evictions_by_year.hvplot(
    x="Year", y="evictions", width=600, height=400, kind="line",
    title="Total Evictions in Philadelphia, 2003 to 2016",
    xformatter="%d"
)
evictions_by_year_plot

### 1.1.5 The number of evictions across Philadelphia

Our tidy data frame is still a GeoDataFrame with a geometry column, so we can visualize the number of evictions for all census tracts. 

Use `hvplot()` to generate a choropleth showing the number of evictions for a specified year, with a widget dropdown to select a given year (or variable name, e.g., `e-16`, `e-15`, etc). 

**Hints** 
- You'll need to use the `groupby` keyword to tell `hvplot` to make a series of maps, with a widget to select between them.
- You will need to specify `dynamic=False` as a keyword argument to the `hvplot()` function. 
- Be sure to specify a `width` and `height` that makes your output map (roughly) square to limit distortions

In [ ]:
evictions_tidy["Year"] = 2000 + evictions_tidy["Date"].str[2:].astype(int)

In [ ]:
import geopandas as gpd
import geoviews as gv
hv.extension("bokeh")
evictions_tidy = gpd.GeoDataFrame(evictions_tidy, geometry="geometry", crs=philly_county.crs)

In [ ]:
import panel as pn

evictions_map = evictions_tidy.hvplot(
    c="evictions",
    groupby="Year",
    kind="polygons",
    dynamic=False,
    widgets={"Year": pn.widgets.Select},
    widget_location="top_left",
    width=600,
    height=600,
    cmap="viridis",
    clabel="Number of evictions",
    line_color="white",
    line_width=0.3,
    xaxis=None,
    yaxis=None,
    toolbar=None,
)

pn.Column(
    pn.pane.Markdown("## Evictions by Census Tract in Philadelphia, 2003 to 2016"),
    evictions_map,
)

## 1.2 Code Violations in Philadelphia

Next, we'll explore data for code violations from the Licenses and Inspections Department of Philadelphia to look for potential correlations with the number of evictions.


### 1.2.1 Load data from 2012 to 2016

L+I violation data for years including 2012 through 2016 (inclusive) is provided in a CSV format in the "data/" folder. 

Load the data using pandas and convert to a GeoDataFrame.

In [ ]:
import geopandas as gpd

philly_code_violations = pd.read_csv("../data/assignment3-data/li_violations.csv")

philly_code_violations = philly_code_violations.dropna(subset=["lat", "lng"])

philly_violations = gpd.GeoDataFrame(
    philly_code_violations,
    geometry=gpd.points_from_xy(
        philly_code_violations["lng"],
        philly_code_violations["lat"],
    ),
    crs="EPSG:4326",
)

In [ ]:
print(philly_violations.crs)
philly_violations["geometry"].head()

### 1.2.2 Trim to specific violation types

There are many different types of code violations (running the `nunique()` function on the `violationdescription` column will extract all of the unique ones). More information on different types of violations can be found [on the City's website](https://www.phila.gov/li/codesandregulations/Pages/codes.aspx).

Below, I've selected 15 types of violations that deal with property maintenance and licensing issues. We'll focus on these violations. The goal is to see if these kinds of violations are correlated spatially with the number of evictions in a given area. 

Use the list of violations given to trim your data set to only include these types.

In [ ]:
violation_types = [
    "INT-PLMBG MAINT FIXTURES-RES",
    "INT S-CEILING REPAIR/MAINT SAN",
    "PLUMBING SYSTEMS-GENERAL",
    "CO DETECTOR NEEDED",
    "INTERIOR SURFACES",
    "EXT S-ROOF REPAIR",
    "ELEC-RECEPTABLE DEFECTIVE-RES",
    "INT S-FLOOR REPAIR",
    "DRAINAGE-MAIN DRAIN REPAIR-RES",
    "DRAINAGE-DOWNSPOUT REPR/REPLC",
    "LIGHT FIXTURE DEFECTIVE-RES",
    "LICENSE-RES SFD/2FD",
    "ELECTRICAL -HAZARD",
    "VACANT PROPERTIES-GENERAL",
    "INT-PLMBG FIXTURES-RES",
]

In [ ]:
philly_violations["violationdescription"].nunique()

In [ ]:
philly_violations.columns

In [ ]:
philly_prop_violations = philly_violations[
    philly_violations["violationdescription"].isin(violation_types)
]
philly_prop_violations.head()

### 1.2.3 Make a hex bin map

The code violation data is **point data**. We can get a quick look at the geographic distribution using `matplotlib` and the `hexbin()` function. Make a hex bin map of the code violations and overlay the census tract outlines. 

**Hints**:
- The eviction data from part 1 was by census tract, so the census tract geometries are available as part of that GeoDataFrame. You can use it to overlay the census tracts on your hex bin map.
- Make sure you convert your GeoDataFrame to a CRS that's better for visualization than plain old 4326.

In [ ]:
violations_proj = philly_prop_violations.to_crs("EPSG:3857")
tracts_proj = evictions_tidy.drop_duplicates(subset="GEOID").to_crs("EPSG:3857")
print(len(tracts_proj))
print(tracts_proj.crs)
print(violations_proj.crs)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 10), layout="constrained")  

# Extract x/y coordinates of the Point objects
xcoords = violations_proj.geometry.x
ycoords = violations_proj.geometry.y

# Edges of the city (this defines minx, maxx, miny, maxy)
minx, miny, maxx, maxy = tracts_proj.total_bounds               

hex_vals = ax.hexbin(
    xcoords, ycoords,
    gridsize=50,
    extent=(minx, maxx, miny, maxy),                              
    cmap="YlOrRd",                                                
    mincnt=1,                                                     
)

# Add the Philly census tracts
tracts_proj.plot(
    ax=ax, facecolor="none", edgecolor="grey", linewidth=0.3     
)

# Map settings
ax.set_xlim(minx, maxx)
ax.set_ylim(miny, maxy)
ax.set_aspect("equal")
ax.set_axis_off()
ax.set_title("Property Maintenance Code Violations in Philadelphia, 2012 to 2016", fontsize=16)

cbar = fig.colorbar(hex_vals, ax=ax, shrink=0.8)
cbar.ax.yaxis.set_label_position("left")
cbar.ax.set_ylabel("Number of violations", fontsize=12, loc="bottom")

plt.show()

### 1.2.4 Spatially join data sets

To do a census tract comparison to our eviction data, we need to find which census tract each of the code violations falls into. Use the `geopandas.sjoin()` function to do just that. 


**Hints**
- You can re-use your eviction data frame, but you will only need the `geometry` column (specifying census tract polygons) and the `GEOID` column (specifying the name of each census tract).
- Make sure both data frames have the same CRS before joining them together!

In [ ]:
compare_evic_voil = gpd.sjoin(
    violations_proj,
    tracts_proj[["GEOID", "geometry"]],
    predicate="within",
    how="left",
)

In [ ]:
print(len(violations_proj), len(compare_evic_voil))
compare_evic_voil["GEOID"].isna().sum()

### 1.2.5 Calculate the number of violations by type per census tract

Next, we'll want to find the number of violations (for each kind) per census tract. You should group the data frame by violation type and census tract name.

The result of this step should be a data frame with three columns: `violationdescription`, `GEOID`, and `N`, where `N` is the number of violations of that kind in the specified census tract.

**Optional: to make prettier plots**

Some census tracts won't have any violations, and they won't be included when we do the above calculation. However, there is a trick to set the values for those census tracts to be zero. After you calculate the sizes of each violation/census tract group, you can run: 

```python
N = N.unstack(fill_value=0).stack().reset_index(name='N')
```
where `N` gives the total size of each of the groups, specified by violation type and census tract name.

See [this StackOverflow post](https://stackoverflow.com/questions/42854801/including-missing-combinations-of-values-in-a-pandas-groupby-aggregation) for more details.

This part is **optional**, but will make the resulting maps a bit prettier.

In [ ]:
violations_by_tract = (
    compare_evic_voil
    .groupby(["violationdescription", "GEOID"], as_index=False)
    .size()
    .rename(columns={"size": "N"})
)
violations_by_tract.head()

In [ ]:
# 1. count violations per type per tract (a Series with a two-level index)
N = compare_evic_voil.groupby(["violationdescription", "GEOID"]).size()

# 2. fill in the missing combinations with 0
violations_by_tract = N.unstack(fill_value=0).stack().reset_index(name="N")

violations_by_tract.head()

### 1.2.6 Merge with census tracts geometries

We now have the number of violations of different types per census tract specified as a regular DataFrame. You can now merge it with the census tract geometries (from your eviction data GeoDataFrame) to create a GeoDataFrame. 

**Hints**
- Use `pandas.merge()` and specify the `on` keyword to be the column holding census tract names. 
- Make sure the result of the merge operation is a GeoDataFrame — you will want the GeoDataFrame holding census tract geometries to be the first argument of the `pandas.merge()` function.

In [ ]:
violations_tract_gdf = pd.merge(
    tracts_proj[["GEOID", "geometry"]],
    violations_by_tract,
    on="GEOID",
)

violations_tract_gdf.head()

### 1.2.7 Interactive choropleths for each violation type

Now, we can use `hvplot()` to create an interactive choropleth for each violation type and add a widget to specify different violation types. 

**Hints** 
- You'll need to use the `groupby` keyword to tell `hvplot` to make a series of maps, with a widget to select different violation types.
- You will need to specify `dynamic=False` as a keyword argument to the `hvplot()` function. 
- Be sure to specify a `width` and `height` that makes your output map (roughly) square to limit distortions

In [ ]:
import panel as pn

violations_map = violations_tract_gdf.hvplot(
    c="N",
    groupby="violationdescription",
    kind="polygons",
    dynamic=False,
    widgets={"violationdescription": pn.widgets.Select},
    widget_location="top_left",
    width=600,
    height=600,
    cmap="YlOrRd",
    clabel="Number of violations",
    line_color="white",
    line_width=0.3,
    xaxis=None,
    yaxis=None,
    toolbar=None,
)

pn.Column(
    pn.pane.Markdown("## Code Violations by Type and Census Tract in Philadelphia, 2012 to 2016"),
    violations_map,
)

## 1.3. A side-by-side comparison

From the interactive maps of evictions and violations, you should notice a lot of spatial overlap. 

As a final step, we'll make a side-by-side comparison to better show the spatial correlations. This will involve a few steps: 

1. Trim the evictions data frame plotted in section 1.1.5 to only include evictions from 2016. 
2. Trim the L+I violations data frame plotted in section 1.2.7 to only include a single violation type (pick whichever one you want!).
3. Use `hvplot()` to make two interactive choropleth maps, one for the data from step 1. and one for the data in step 2.
4. Show these two plots side by side (one row and 2 columns) using the syntax for combining charts.

**Note:** since we selected a single year and violation type, you won't need to use the `groupby=` keyword here.

In [ ]:
evictions_2016 = evictions_tidy[evictions_tidy["Year"] == 2016].to_crs("EPSG:3857")

violation_plumbing = violations_tract_gdf[
    violations_tract_gdf["violationdescription"] == "PLUMBING SYSTEMS-GENERAL"
] 

In [ ]:
evictions_plot = evictions_2016.hvplot(
    c="evictions",
    kind="polygons",
    width=500,
    height=500,
    cmap="viridis",
    clabel="Number of evictions",
    line_color="white",
    line_width=0.3,
    xaxis=None,
    yaxis=None,
    toolbar=None,
    title="Evictions in 2016",
)

violations_plot = violation_plumbing.hvplot(
    c="N",
    kind="polygons",
    width=500,
    height=500,
    cmap="YlOrRd",
    clabel="Number of violations",
    line_color="white",
    line_width=0.3,
    xaxis=None,
    yaxis=None,
    toolbar=None,
    title="Plumbing Violations",
)

In [ ]:
(evictions_plot + violations_plot).cols(2).opts(toolbar=None)

## Evictions and Building Violations Cluster in the Same Philadelphia Neighborhoods

Evictions and building code violations are strongly linked across Philadelphia's census tracts. The correlation between the two is **0.71**, and **57%** of the tracts with the most evictions also rank among the tracts with the most violations. That is more than twice the 25% we would expect by chance. The top 25% of violation tracts alone hold **44%** of all evictions from 2012 to 2016, concentrated in North Philadelphia, Kensington and Frankford, and West and Southwest Philadelphia.

This suggests that poor housing conditions and displacement pressure tend to hit the same neighborhoods, which fits the idea that evictions can follow tenant's complaints about conditions. These places may need targeted housing inspections and tenant support.

*Note: these are raw counts, so larger tracts naturally have more of both, and a correlation does not show that one causes the other.*

## Part 2: Exploring the NDVI in Philadelphia

In this part, we'll explore the NDVI in Philadelphia a bit more. This part will include two parts:

1. We'll compare the median NDVI within the city limits and the immediate suburbs
1. We'll calculate the  NDVI around street trees in the city.

## 2.1 Comparing the NDVI in the city and the suburbs

### 2.1.1 Load Landsat data for Philadelphia

Use rasterio to load the landsat data for Philadelphia (available in the "data/" folder)

In [ ]:
import rasterio as rio

landsat = rio.open("../data/assignment3-data/landsat8_philly.tif")
landsat
print(landsat.crs)
print(landsat.count)
print(landsat.indexes)

### 2.1.2 Separating the city from the suburbs

Create two polygon objects, one for the city limits and one for the suburbs. To calculate the suburbs polygon, we will take everything outside the city limits but still within the bounding box. 

* The city limits are available in the "data/" folder.
* To calculate the suburbs polygon, the "envelope" attribute of the city limits geometry will be useful.
* You can use geopandas' [geometric manipulation](https://geopandas.org/en/stable/docs/user_guide/geometric_manipulations.html) functionality to calculate the suburbs polygon from the city limits polygon and the envelope polygon.

In [ ]:
city = gpd.read_file("../data/assignment3-data/City_Limits.geojson").to_crs(landsat.crs)
city_polygon = city.union_all()                              # city limits as one shape
suburbs_polygon = city_polygon.envelope.difference(city_polygon)   # box minus city

### 2.1.3 Mask and calculate the NDVI for the city and the suburbs

Using the two polygons from the last section, use rasterio's mask functionality to create two masked arrays from the landsat data, one for the city and one for the suburbs. 

For each masked array, calculate the NDVI.

In [ ]:
from rasterio.mask import mask

# Part A: mask the raster to the city
city_masked, city_transform = mask(
    landsat,
    [city_polygon],
    crop=True,
    filled=False,
)

# Part B: NDVI
red = city_masked[3].astype(float)    # band 4 (red); Pythons numbering 
nir = city_masked[4].astype(float)    # band 5 (NIR); Pythons numbering 

ndvi_city = (nir - red) / (nir + red)

In [ ]:
# Part B: mask the raster to Suburbs
suburbs_masked, suburbs_transform = mask(
    landsat,
    [suburbs_polygon],
    crop=True,
    filled=False,
)

# Part B: NDVI
red_s = suburbs_masked[3].astype(float)    # band 4 (red); Pythons numbering 
nir_s = suburbs_masked[4].astype(float)    # band 5 (NIR); Pythons numbering 

ndvi_suburbs = (nir_s - red_s) / (nir_s + red_s)

### 2.1.4 Calculate the median NDVI within the city and within the suburbs

* Calculate the median value from your NDVI arrays for the city and suburbs
* Numpy's `nanmedian` function will be useful for ignoring NaN elements
* Print out the median values. Which has a higher NDVI: the city or suburbs?

In [ ]:
import numpy as np 

median_city = np.nanmedian(ndvi_city.filled(np.nan))
median_suburbs = np.nanmedian(ndvi_suburbs.filled(np.nan))

print(f"Median NDVI in the city: {median_city: .3f}")
print(f"Median NDVI in the suburbs: {median_suburbs: .3f}")

In [ ]:
from IPython.display import Markdown, display
if median_city > median_suburbs:
    display(Markdown("**The city has the higher median NDVI**"))
else:
    display(Markdown("**The suburbs have higher median NDVI**"))

## 2.2 Calculating the NDVI for Philadelphia's street trees

### 2.2.1 Load the street tree data

The data is available in the "data/" folder. It has been downloaded from [OpenDataPhilly](https://www.opendataphilly.org/dataset/ppr-tree-canopy). It contains the locations of abot 2,500 street trees in Philadelphia.

In [ ]:
philly_trees = gpd.read_file("../data/assignment3-data/ppr_tree_canopy_points_2015.geojson")
philly_trees.head()
philly_trees = philly_trees.to_crs(landsat.crs)

### 2.2.2 Calculate the NDVI values at the locations of the street trees

* Use the rasterstats package to calculate the NDVI values at the locations of the street trees.
* Since these are point geometries, you can calculate either the median or the mean statistic (only one pixel will contain each point).

In [ ]:
# band 4 (red), This uses people's numbering
red_trees = landsat.read(4).astype(float) 
# band 5 (NIR)
nir_trees = landsat.read(5).astype(float)

ndvi_trees = (nir_trees - red_trees) / (nir_trees + red_trees)

In [ ]:
from rasterstats import zonal_stats

tree_stats = zonal_stats(
    philly_trees,
    ndvi_trees,
    affine = landsat.transform,
    stats = ["mean"],
    nodata=-999,
)

### 2.2.3 Plotting the results

Make two plots of the results:

1. A histogram of the NDVI values, using matplotlib's `hist` function. Include a vertical line that marks the NDVI = 0 threshold
1. A plot of the street tree points, colored by the NDVI value, using geopandas' `plot` function. Include the city limits boundary on your plot.

The figures should be clear and well-styled, with for example, labels for axes, legends, and clear color choices. 

In [ ]:
philly_trees["NDVI"] = [s["mean"] for s in tree_stats]
philly_trees = philly_trees.dropna(subset=["NDVI"])

In [ ]:
counts, edges = np.histogram(philly_trees["NDVI"], bins=25)

hist = hv.Histogram((edges, counts), kdims="NDVI", vdims="Trees").opts(
    color="seagreen",
    line_color="white",
    xlabel="NDVI (Normalized Difference Vegetation Index)",
    ylabel="Number of street trees",
    title="NDVI at Philadelphia street tree locations",
    width=700, height=450,
    hover_tooltips=[
        ("NDVI from", "@left{0.000}"),
        ("to", "@right{0.000}"),
        ("Trees", "@Trees"),
    ],
)

zero_line = hv.VLine(0).opts(color="red", line_dash="dashed", line_width=2)

hist * zero_line

**What is NDVI?**
NDVI is a satellite "greenness score" that shows how much healthy vegetation is in a spot. Healthy plants reflect a lot of near-infrared light and absorb red light, and NDVI compares the two.

**How to read it**
- **Below 0:** water, bare ground, or no vegetation
- **0 to 0.2:** pavement, rooftops, or sparse vegetation
- **0.2 to 0.5:** moderate greenery, like grass or thin tree cover
- **Above 0.5:** dense, healthy vegetation

**What the histogram shows**
Each bar counts how many street trees fall in an NDVI range. The red dashed line marks NDVI = 0. Trees to its right have some greenery around them. Trees at or below it sit in areas that look like bare ground or concrete from space.

In [ ]:
import logging
import param
param.parameterized.get_logger().setLevel(logging.ERROR)
logging.getLogger("param").setLevel(logging.ERROR)

# ... then the rest of your map code

In [ ]:
# ---------- 1. Prepare the data ----------
# Tracts reprojected to match the city and trees (EPSG:32618)
tracts_plot = tracts_proj.to_crs(city.crs)[["geometry"]]

# Trees as a plain table: x, y, NDVI
trees_df = pd.DataFrame({
    "x": philly_trees.geometry.x,
    "y": philly_trees.geometry.y,
    "NDVI": philly_trees["NDVI"],
})

# Sort each tree into an NDVI range
labels = ["Below 0", "0 to 0.1", "0.1 to 0.2", "0.2 to 0.3", "0.3 to 0.4", "Above 0.4"]
colors = ["#d73027", "#fc8d59", "#fee08b", "#d9ef8b", "#91cf60", "#1a9850"]  # red to green

trees_df["NDVI range"] = pd.cut(
    trees_df["NDVI"],
    bins=[-10, 0, 0.1, 0.2, 0.3, 0.4, 10],
    labels=labels,
)

# ---------- 2. Build the layers ----------
# Faint census tract boundaries
tract_layer = tracts_plot.hvplot(kind="polygons", hover=False).opts(
    fill_alpha=0, line_color="gray", line_alpha=0.3, line_width=0.5
)

# Subtle dashed city outline
city_layer = city[["geometry"]].hvplot(kind="polygons", hover=False).opts(
    fill_alpha=0, line_color="black", line_alpha=0.6,
    line_dash="dashed", line_width=1.2
)

# One layer of points per NDVI range (gives one legend entry each)
tree_layers = [
    trees_df[trees_df["NDVI range"] == lab].hvplot.scatter(
        x="x", y="y", color=col, size=5, alpha=0.85, label=lab,
        hover=False,
    )
    for lab, col in zip(labels, colors)
]

# ---------- 3. Styling: bold title, bold legend title ----------
def style(plot, element):
    plot.state.title.text_font_style = "bold"
    legend = plot.state.legend[0]
    legend.title = "NDVI"
    legend.title_text_font_style = "bold"

# ---------- 4. Stack the layers and show ----------
(tract_layer * city_layer * hv.Overlay(tree_layers)).opts(
    title="NDVI at Street Tree Locations in Philadelphia",
    hooks=[style],
    data_aspect=1,
    frame_height=750,
    toolbar=None,
    tools=["wheel_zoom", "pan"],
    active_tools=["wheel_zoom"],
    legend_position="bottom_right",
    xaxis=None, yaxis=None,
)

## Street Trees Need Care: Low Greenness Across Philadelphia

Most of Philadelphia's street trees sit in places with low greenness. About **80%** of the trees have an NDVI between 0 and 0.3, and the highest value in the city is only **0.59**. Even the greenest locations barely reach the range of dense, healthy vegetation (above 0.5).

This suggests the overall picture for street trees is not very promising. Many trees may be struggling, or they may be sitting in heavily built-up areas, and they may need extra care and attention such as watering, better soil, and more greenery nearby. The map shows the low-NDVI trees (red) clustering in the denser parts of the city, which are the places where this support would matter most.

*Note: Landsat pixels are 30 m wide, so each value also reflects the pavement and buildings around a tree, not only the tree itself.*